In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import dask.dataframe as dd

In [69]:
from sklearn import set_config
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_absolute_percentage_error

In [3]:
# data path

df_jan_path = r"..\data\raw\yellow_tripdata_2016-01.csv"
df_feb_path = r"..\data\raw\yellow_tripdata_2016-02.csv"
df_mar_path = r"..\data\raw\yellow_tripdata_2016-03.csv"

# loading the data

df_jan = dd.read_csv(df_jan_path,assume_missing=True,usecols=['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'],parse_dates=["tpep_pickup_datetime"])

df_feb = dd.read_csv(df_feb_path,assume_missing=True,usecols=['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'],parse_dates=["tpep_pickup_datetime"])

df_mar = dd.read_csv(df_mar_path,assume_missing=True,usecols=['trip_distance', 'tpep_pickup_datetime', 'pickup_longitude',
       'pickup_latitude','dropoff_longitude', 'dropoff_latitude', 'fare_amount'],parse_dates=["tpep_pickup_datetime"])

In [4]:
df_jan

,tpep_pickup_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
npartitions=26,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [5]:
df = dd.concat([df_jan,df_feb,df_mar],axis=0)

df

,tpep_pickup_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
npartitions=82,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [6]:
# set the values of coordinates

min_latitude = 40.60
max_latitude = 40.85

min_longitude = -74.05
max_longitude = -73.70

min_fare_amount_val = 0.50
max_fare_amount_val = 81.0

min_trip_distance_val = 0.25
max_trip_distance_val = 24.43

In [7]:
df.columns

Index(['tpep_pickup_datetime', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
       'fare_amount'],
      dtype='object')

In [8]:
pickup_latitude_bool = df['pickup_latitude'].between(min_latitude,max_latitude,inclusive='both')
pickup_longitude_bool = df['pickup_longitude'].between(min_longitude,max_longitude,inclusive='both')
dropoff_latitude_bool = df['dropoff_latitude'].between(min_latitude,max_latitude,inclusive='both')
dropoff_longitude_bool = df['dropoff_longitude'].between(min_longitude,max_longitude,inclusive='both')


In [9]:
df = df[pickup_latitude_bool & pickup_longitude_bool & dropoff_latitude_bool & dropoff_longitude_bool]

df

,tpep_pickup_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
npartitions=82,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [10]:
trip_distance_bool = df['trip_distance'].between(min_trip_distance_val,max_trip_distance_val,inclusive='both')
fare_amount_bool = df['fare_amount'].between(min_fare_amount_val,max_fare_amount_val,inclusive='both')

In [11]:
df = df[trip_distance_bool & fare_amount_bool]

df

,tpep_pickup_datetime,trip_distance,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,fare_amount
npartitions=82,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [12]:
df = df.drop(columns=['trip_distance', 'dropoff_longitude', 'dropoff_latitude', 'fare_amount'])

df

,tpep_pickup_datetime,pickup_longitude,pickup_latitude
npartitions=82,,,
,datetime64[ns],float64,float64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [13]:
df = df.compute()

df

,tpep_pickup_datetime,pickup_longitude,pickup_latitude
0,2016-01-01 00:00:00,-73.990372,40.734695
1,2016-01-01 00:00:00,-73.980782,40.729912
2,2016-01-01 00:00:00,-73.984550,40.679565
3,2016-01-01 00:00:00,-73.993469,40.718990
4,2016-01-01 00:00:00,-73.960625,40.781330
...,...,...,...
420269,2016-03-31 21:43:11,-73.790565,40.644451
420270,2016-03-20 08:45:16,-73.788055,40.641483
420271,2016-03-20 08:59:21,-73.789154,40.646736
420273,2016-03-26 03:02:32,-73.977356,40.774471


In [14]:
df.shape

(33234199, 3)

In [15]:
# save the dataframe

save_path = r"../data/interim/processing_data.csv"

df.to_csv(save_path,index=False)

In [16]:
data_path = r"../data/interim/processing_data.csv"

df_iter = pd.read_csv(data_path,chunksize=100000,usecols=["pickup_latitude","pickup_longitude"])

df_iter

In [17]:
# train the scaler

scaler = StandardScaler()

for chunk in df_iter:

    # partial fit the data
    scaler.partial_fit(chunk)

scaler

,copy,True
,with_mean,True
,with_std,True


In [18]:
df_iter = pd.read_csv(data_path,chunksize=100000,usecols=["pickup_latitude","pickup_longitude"])

# mini batch object
mini_batch = MiniBatchKMeans(n_clusters=30,n_init='auto',init='k-means++',batch_size=1024)

for chunk in df_iter:

    # scale the chunk
    scaled_chunk = scaler.transform(chunk)

    # train the model
    mini_batch.partial_fit(scaled_chunk)

mini_batch

,n_clusters,30
,init,'k-means++'
,max_iter,100
,batch_size,1024
,verbose,0
,compute_labels,True
,random_state,None
,tol,0.0
,max_no_improvement,10
,init_size,None
,n_init,'auto'


In [19]:
# centroid of the model

mini_batch.cluster_centers_

array([[-0.52315932, -0.01847474],
       [-0.31104667, -0.89441698],
       [ 5.15231186, -3.83966082],
       [ 0.32293407,  1.89107954],
       [ 0.4846414 ,  0.8627925 ],
       [ 1.27897687,  0.2345994 ],
       [-0.3044919 , -2.26036299],
       [-0.31575086, -0.5510745 ],
       [ 0.27393872,  0.46256015],
       [ 1.07579466, -2.01258782],
       [-0.1893733 ,  0.99908398],
       [-0.99511992, -1.58524942],
       [ 2.3074533 , -0.37347121],
       [-0.17067704, -0.17150576],
       [ 0.03941904,  1.41108185],
       [-0.93039476, -1.20335208],
       [-0.38114952, -3.0063351 ],
       [-0.27369973,  0.59785822],
       [ 2.8556847 ,  0.76443615],
       [ 0.80756788,  2.50047125],
       [-0.04465806,  0.35649845],
       [ 0.61870941, -3.29341251],
       [-0.81129042, -0.3732582 ],
       [-0.52279058, -1.12915022],
       [-0.50272643, -0.34979044],
       [-0.70636017, -0.78343954],
       [-0.41884467,  0.28364711],
       [ 0.6382176 ,  1.25500097],
       [ 0.50906352,

In [20]:
# original centroid of the model

og_centroid = scaler.inverse_transform(mini_batch.cluster_centers_)

og_centroid

array([[-73.99328915,  40.75056828],
       [-73.98549351,  40.72672279],
       [-73.78470213,  40.64654537],
       [-73.96219321,  40.80255146],
       [-73.95625008,  40.77455873],
       [-73.92705637,  40.75745763],
       [-73.9852526 ,  40.68953808],
       [-73.9856664 ,  40.73606949],
       [-73.96399391,  40.76366334],
       [-73.9345238 ,  40.69628319],
       [-73.98102172,  40.77826895],
       [-74.01063482,  40.70791648],
       [-73.88925743,  40.74090433],
       [-73.98033459,  40.74640237],
       [-73.97261306,  40.78948463],
       [-74.00825602,  40.71831275],
       [-73.98806995,  40.66923073],
       [-73.98412092,  40.76734651],
       [-73.86910862,  40.77188121],
       [-73.94438177,  40.81914073],
       [-73.97570309,  40.76077605],
       [-73.95132277,  40.66141571],
       [-74.00387865,  40.74091013],
       [-73.9932756 ,  40.72033272],
       [-73.99253819,  40.74154898],
       [-74.00002221,  40.72974389],
       [-73.98945534,  40.75879284],
 

In [21]:
location_df = df.iloc[:,1:]

location_df

,pickup_longitude,pickup_latitude
0,-73.990372,40.734695
1,-73.980782,40.729912
2,-73.984550,40.679565
3,-73.993469,40.718990
4,-73.960625,40.781330
...,...,...
420269,-73.790565,40.644451
420270,-73.788055,40.641483
420271,-73.789154,40.646736
420273,-73.977356,40.774471


In [22]:
scaled_location_df = scaler.transform(location_df)

scaled_location_df

array([[-0.44377822, -0.60154915],
       [-0.18283861, -0.77727142],
       [-0.28538767, -2.62669931],
       ...,
       [ 5.03117896, -3.83265296],
       [-0.08963107,  0.85958016],
       [ 4.77729738, -3.89290861]], shape=(33234199, 2))

In [23]:
# get the cluster prediction

cluster_prediction = mini_batch.predict(scaled_location_df)

cluster_prediction

array([ 7,  1,  6, ...,  2, 10,  2], shape=(33234199,), dtype=int32)

In [24]:
cluster_prediction.shape

(33234199,)

In [25]:
df['region'] = cluster_prediction

df

,tpep_pickup_datetime,pickup_longitude,pickup_latitude,region
0,2016-01-01 00:00:00,-73.990372,40.734695,7
1,2016-01-01 00:00:00,-73.980782,40.729912,1
2,2016-01-01 00:00:00,-73.984550,40.679565,6
3,2016-01-01 00:00:00,-73.993469,40.718990,23
4,2016-01-01 00:00:00,-73.960625,40.781330,4
...,...,...,...,...
420269,2016-03-31 21:43:11,-73.790565,40.644451,2
420270,2016-03-20 08:45:16,-73.788055,40.641483,2
420271,2016-03-20 08:59:21,-73.789154,40.646736,2
420273,2016-03-26 03:02:32,-73.977356,40.774471,10


In [26]:
# drop the lat long columns

time_series_data = df.drop(columns=['pickup_longitude','pickup_latitude'])

time_series_data

,tpep_pickup_datetime,region
0,2016-01-01 00:00:00,7
1,2016-01-01 00:00:00,1
2,2016-01-01 00:00:00,6
3,2016-01-01 00:00:00,23
4,2016-01-01 00:00:00,4
...,...,...
420269,2016-03-31 21:43:11,2
420270,2016-03-20 08:45:16,2
420271,2016-03-20 08:59:21,2
420273,2016-03-26 03:02:32,10


In [36]:
time_series_data.isnull().sum()

region    0
dtype: int64

In [28]:
time_series_data.dtypes

tpep_pickup_datetime    datetime64[ns]
region                           int32
dtype: object

In [27]:
# save the time series data

save_path = r"..\data\interim\time_series.csv"

df.to_csv(save_path,index=False)

In [31]:
time_series_data.set_index('tpep_pickup_datetime',inplace=True)

In [32]:
time_series_data

,region
tpep_pickup_datetime,
2016-01-01 00:00:00,7
2016-01-01 00:00:00,1
2016-01-01 00:00:00,6
2016-01-01 00:00:00,23
2016-01-01 00:00:00,4
...,...
2016-03-31 21:43:11,2
2016-03-20 08:45:16,2
2016-03-20 08:59:21,2


In [35]:
# grouping based on regions

region_grp = time_series_data.groupby('region')

region_grp

In [40]:
resampled_data = region_grp['region'].resample('15min').count()

resampled_data

region  tpep_pickup_datetime
0       2016-01-01 00:00:00     197
        2016-01-01 00:15:00     302
        2016-01-01 00:30:00     326
        2016-01-01 00:45:00     345
        2016-01-01 01:00:00     333
                               ... 
29      2016-03-31 22:45:00     451
        2016-03-31 23:00:00     385
        2016-03-31 23:15:00     314
        2016-03-31 23:30:00     288
        2016-03-31 23:45:00     270
Name: region, Length: 262080, dtype: int64

In [47]:
resampled_data.name = 'total_pickups'

resampled_data

region  tpep_pickup_datetime
0       2016-01-01 00:00:00     197
        2016-01-01 00:15:00     302
        2016-01-01 00:30:00     326
        2016-01-01 00:45:00     345
        2016-01-01 01:00:00     333
                               ... 
29      2016-03-31 22:45:00     451
        2016-03-31 23:00:00     385
        2016-03-31 23:15:00     314
        2016-03-31 23:30:00     288
        2016-03-31 23:45:00     270
Name: total_pickups, Length: 262080, dtype: int64

In [55]:
type(resampled_data)

pandas.core.series.Series

In [63]:
resampled_data = resampled_data.reset_index(level=0)

resampled_data

,region,total_pickups
tpep_pickup_datetime,,
2016-01-01 00:00:00,0,197
2016-01-01 00:15:00,0,302
2016-01-01 00:30:00,0,326
2016-01-01 00:45:00,0,345
2016-01-01 01:00:00,0,333
...,...,...
2016-03-31 22:45:00,29,451
2016-03-31 23:00:00,29,385
2016-03-31 23:15:00,29,314


In [66]:
(resampled_data['total_pickups'] == 0).sum()

np.int64(4092)

In [67]:
epsilon_val = 10

resampled_data.replace({'total_pickups':{0:epsilon_val}},inplace=True)

resampled_data

,region,total_pickups
tpep_pickup_datetime,,
2016-01-01 00:00:00,0,197
2016-01-01 00:15:00,0,302
2016-01-01 00:30:00,0,326
2016-01-01 00:45:00,0,345
2016-01-01 01:00:00,0,333
...,...,...
2016-03-31 22:45:00,29,451
2016-03-31 23:00:00,29,385
2016-03-31 23:15:00,29,314


In [68]:
(resampled_data['total_pickups'] == 0).sum()

np.int64(0)

## Smoothing

### Moving Average

In [88]:
def calculating_best_window(windows):

    for window in windows:

        # first window-1 values will be NaN for moving average
        ind = window-1

        # calculating simple moving average and filtering out first window-1 values
        y_pred = resampled_data['total_pickups'].rolling(window=window).mean().values[ind:]

        # original pickups skipping first window-1 values
        y = resampled_data['total_pickups'].values[ind:]

        # calculating error
        error = mean_absolute_percentage_error(y,y_pred)

        print(f"For window value {window}, the MAPE is {error:.2f}")

In [89]:
window_values = list(range(3,11,1))

window_values

[3, 4, 5, 6, 7, 8, 9, 10]

In [90]:
calculating_best_window(window_values)

For window value 3, the MAPE is 0.20
For window value 4, the MAPE is 0.25
For window value 5, the MAPE is 0.29
For window value 6, the MAPE is 0.33
For window value 7, the MAPE is 0.37
For window value 8, the MAPE is 0.41
For window value 9, the MAPE is 0.45
For window value 10, the MAPE is 0.50


- lowest error is 20% for window value 3

## Exponentially Weighted Moving Average

In [93]:
resampled_data['total_pickups'].ewm(alpha=0.9).mean()

tpep_pickup_datetime
2016-01-01 00:00:00    197.000000
2016-01-01 00:15:00    292.454545
2016-01-01 00:30:00    322.675676
2016-01-01 00:45:00    342.769577
2016-01-01 01:00:00    333.976870
                          ...    
2016-03-31 22:45:00    450.508036
2016-03-31 23:00:00    391.550804
2016-03-31 23:15:00    321.755080
2016-03-31 23:30:00    291.375508
2016-03-31 23:45:00    272.137551
Name: total_pickups, Length: 262080, dtype: float64

In [99]:
def calculate_best_smmothing_value(alphas):
    # y actual
    y = resampled_data['total_pickups'].values

    for alpha in alphas:
        # y pred
        y_pred = resampled_data['total_pickups'].ewm(alpha=alpha).mean()
        # calculating error
        error = mean_absolute_percentage_error(y,y_pred)
        print(f"For Smoothing value {alpha:.1f}, the MAPE is {error:.2f}")

In [100]:
alphas = np.arange(0.5,1,0.1)
alphas

array([0.5, 0.6, 0.7, 0.8, 0.9])

In [101]:
calculate_best_smmothing_value(alphas)

For Smoothing value 0.5, the MAPE is 0.16
For Smoothing value 0.6, the MAPE is 0.12
For Smoothing value 0.7, the MAPE is 0.09
For Smoothing value 0.8, the MAPE is 0.06
For Smoothing value 0.9, the MAPE is 0.03


- Lowest error is 3% for alpha=0.9

### Dataset with Smoothing value

In [110]:
resampled_data['avg_pickups'] = resampled_data['total_pickups'].ewm(alpha=0.9).mean().round()

resampled_data

,region,total_pickups,avg_pickups
tpep_pickup_datetime,,,
2016-01-01 00:00:00,0,197,197.0
2016-01-01 00:15:00,0,302,292.0
2016-01-01 00:30:00,0,326,323.0
2016-01-01 00:45:00,0,345,343.0
2016-01-01 01:00:00,0,333,334.0
...,...,...,...
2016-03-31 22:45:00,29,451,451.0
2016-03-31 23:00:00,29,385,392.0
2016-03-31 23:15:00,29,314,322.0


### EWMA with alpha = 0.4 : (2/(n + 1)), n = 4 : previous 4 time stamp

In [116]:
resampled_data.drop(columns='avg_pickups',inplace=True)

resampled_data

,region,total_pickups
tpep_pickup_datetime,,
2016-01-01 00:00:00,0,197
2016-01-01 00:15:00,0,302
2016-01-01 00:30:00,0,326
2016-01-01 00:45:00,0,345
2016-01-01 01:00:00,0,333
...,...,...
2016-03-31 22:45:00,29,451
2016-03-31 23:00:00,29,385
2016-03-31 23:15:00,29,314


In [125]:
resampled_data['avg_pickups'] = resampled_data['total_pickups'].ewm(alpha=0.4).mean().round()

resampled_data

,region,total_pickups,avg_pickups
tpep_pickup_datetime,,,
2016-01-01 00:00:00,0,197,197.0
2016-01-01 00:15:00,0,302,263.0
2016-01-01 00:30:00,0,326,295.0
2016-01-01 00:45:00,0,345,318.0
2016-01-01 01:00:00,0,333,324.0
...,...,...,...
2016-03-31 22:45:00,29,451,459.0
2016-03-31 23:00:00,29,385,430.0
2016-03-31 23:15:00,29,314,383.0


In [126]:
# save the resampled data

resampled_data_save_path = r"..\data\interim\final_data.csv"

resampled_data.to_csv(resampled_data_save_path,index=True)